## 2. Melakukan Tfidf & Word Embedding

In [14]:
!pip install Sastrawi

In [15]:
!pip install gensim

# --- Import Library ---

In [16]:
import pandas as pd
import re
import string
import nltk
from nltk.tokenize import word_tokenize
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

from sklearn.feature_extraction.text import TfidfVectorizer
from gensim.models import Word2Vec

nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

1. Dataset

In [17]:
def load_data(filepath="kompas_articles_new.csv"):
    df = pd.read_csv(filepath)

    if "judul" in df.columns and "isi" in df.columns:
        df["berita"] = df["judul"].astype(str) + " " + df["isi"].astype(str)
    elif "berita" in df.columns:
        df["berita"] = df["berita"].astype(str)
    else:
        df["berita"] = df.apply(lambda row: " ".join([str(x) for x in row]), axis=1)

    return df

In [18]:
df = load_data("kompas_articles_new.csv")

2. Prepocessing

In [19]:
stemmer = StemmerFactory().create_stemmer()
stop_factory = StopWordRemoverFactory()
stopwords = set(stop_factory.get_stop_words())

def clean_text(text):
    text = text.lower()
    text = re.sub(r"\d+", " ", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    tokens = word_tokenize(text)
    tokens = [stemmer.stem(t) for t in tokens if t not in stopwords and len(t) > 2]
    return " ".join(tokens)

def preprocess_data(df):
    df["clean"] = df["berita"].apply(clean_text)
    return df[["berita", "clean"]]

In [20]:
df_clean = preprocess_data(df)
display(df_clean.head())

,berita,clean
0,Jokowi Kenakan Pakaian Adat Betawi di Sidang T...,jokowi kena pakai adat betawi sidang tahun akh...
1,KPU Tegaskan Pemilih Tak Terdaftar di DPT Bisa...,kpu tegas pilih tak daftar dpt nyoblos begini ...
2,Warga Sebut Ada Benda Serupa Jimat pada Mayat ...,warga sebut benda rupa jimat mayat sarung pamu...
3,Polisi Menganiaya Mereka yang Cinta Damai dan ...,polisi aniaya cinta damai bercitacita mulia ja...
4,"Airlangga Hartarto Mundur dari Ketum, Golkar B...",airlangga hartarto mundur tum golkar bantah ko...


3. TFidf

In [21]:
def compute_tfidf(df):
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(df["clean"])

    tfidf_df = pd.DataFrame(
        tfidf_matrix.toarray(),
        columns=vectorizer.get_feature_names_out()
    )
    tfidf_df.index = [f"Dokumen_{i+1}" for i in range(len(df))]

    print("Shape TF-IDF:", tfidf_matrix.shape)
    return tfidf_df, tfidf_matrix

In [22]:
tfidf_df, tfidf_matrix = compute_tfidf(df)
display(tfidf_df.head())

Shape TF-IDF: (128, 2717)


,abadi,abai,abang,abdul,abdullah,abdurrahman,abetnego,abituren,absen,acara,...,youtube,youtubekompas,youtubekompascom,yuda,yudhoyono,yuwono,zainudin,zaman,zulhas,zulkifli
Dokumen_1,0.0,0.0,0.0,0.0,0.0,0.0,0.166888,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0
Dokumen_2,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0
Dokumen_3,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.087124,0.0,0.0
Dokumen_4,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.097464,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0
Dokumen_5,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0


4. Melakukan Word Embedding (CBOW)

A. Bikin fungsi training model Word2Vec

In [23]:
from gensim.models import Word2Vec

def train_word2vec(df, vector_size=100, window=5, min_count=2, sg=0):
    corpus = [row.split() for row in df['clean'].values]
    model = Word2Vec(
        sentences=corpus,
        vector_size=vector_size,
        window=window,
        min_count=min_count,
        sg=sg
    )
    return model

B. Buat fungsi explore_word2vec()

In [24]:
def explore_word2vec(model, word="indonesia", similar_to="ekonomi"):
    print("Shape Word Embedding (jumlah_kata, dimensi):", model.wv.vectors.shape)

    # vektor kata
    if word in model.wv:
        vec = model.wv[word]
        vec_df = pd.DataFrame(vec, columns=[f"Nilai Vektor ({word})"])
    else:
        vec_df = pd.DataFrame([f"Kata '{word}' tidak ada di vocab"], columns=["Info"])

    # kata mirip
    if similar_to in model.wv:
        sim_df = pd.DataFrame(model.wv.most_similar(similar_to, topn=5),
                              columns=["Kata", "Skor Similaritas"])
    else:
        sim_df = pd.DataFrame([f"Kata '{similar_to}' tidak ada di vocab"], columns=["Info"])

    return vec_df, sim_df

In [25]:
model = train_word2vec(df, vector_size=100, window=5, min_count=1, sg=0)

In [26]:
vec_df, sim_df = explore_word2vec(model, word="indonesia", similar_to="negara")
display(vec_df.head())
display(sim_df)
print("ekonomi" in model.wv.key_to_index)
print(list(model.wv.index_to_key)[:50])

Shape Word Embedding (jumlah_kata, dimensi): (2717, 100)


,Nilai Vektor (indonesia)
0,-0.008543
1,0.007640
2,0.010667
3,-0.009089
4,-0.004661


,Kata,Skor Similaritas
0,kata,0.869015
1,sebut,0.839894
2,jakarta,0.837914
3,warga,0.835455
4,jalan,0.832529


True
['jakarta', 'kompascom', 'sukses', 'nasional', 'baca', 'jadi', 'kata', 'partai', 'httpsnasionalkompascomread', 'sebut', 'pilkada', 'tak', 'ketua', 'calon', 'sampai', 'laku', 'anies', 'kota', 'minta', 'httpsmegapolitankompascomread', 'pilih', 'megapolitan', 'presiden', 'indonesia', 'ujar', 'dpr', 'warga', 'orang', 'umum', 'politik', 'jalan', 'pdip', 'bekas', 'depok', 'wakil', 'duga', 'kepala', 'gubernur', 'beri', 'paus', 'dukung', 'golkar', 'prabowo', 'hari', 'depan', 'nyata', 'fransiskus', 'negara', 'daerah', 'menteri']


C. Menentukan masing-masing Kategori

In [28]:
from collections import Counter
import pandas as pd
from IPython.display import display
print("--- ANALISIS FREKUENSI KATA BERDASARKAN KATEGORI ---")

# Ambil daftar kategori unik (seharusnya Nasional dan Megapolitan)
categories = df['Kategori'].unique()

for category in categories:
    # 1. Filter DataFrame untuk kategori saat ini
    df_filtered = df[df['Kategori'] == category]

    # 2. Gabungkan semua token dari semua berita di kategori ini
    all_tokens_category = [
        token
        for sublist in df_filtered['clean']
        if sublist is not None
        for token in sublist.split() # Split the cleaned text into tokens
    ]


    # 3. Hitung frekuensi kata
    freq_category = Counter(all_tokens_category)

    # 4. Urutkan semua kata unik
    all_unique_words = freq_category.items()
    sorted_words = sorted(all_unique_words, key=lambda x: (-x[1], x[0]))

    # Tampilkan Hasil
    print(f"\n========================================================")
    print(f"KATEGORI: {category.upper()}")
    print(f"Total Berita: {len(df_filtered)}")
    print(f"Total Kata Unik: {len(sorted_words)}")
    print(f"========================================================")

    # Tampilkan 10 kata paling sering
    top_10 = sorted_words[:10]

    # Buat DataFrame untuk tampilan yang rapi
    top_10_df = pd.DataFrame(top_10, columns=['Kata', 'Frekuensi'])
    display(top_10_df)

    print("-" * 50)

--- ANALISIS FREKUENSI KATA BERDASARKAN KATEGORI ---

KATEGORI: NASIONAL
Total Berita: 80
Total Kata Unik: 1945


,Kata,Frekuensi
0,jakarta,140
1,nasional,96
2,kompascom,83
3,sukses,81
4,httpsnasionalkompascomread,80
5,partai,75
6,kata,62
7,jadi,58
8,pilkada,53
9,baca,49


--------------------------------------------------

KATEGORI: MEGAPOLITAN
Total Berita: 48
Total Kata Unik: 1401


,Kata,Frekuensi
0,jakarta,115
1,kompascom,50
2,httpsmegapolitankompascomread,48
3,megapolitan,48
4,sukses,48
5,kota,42
6,baca,39
7,bekas,37
8,warga,36
9,depok,32


--------------------------------------------------
